In [ ]:
%run supportvectors-common.ipynb

# Introduction to Google Agent Development Kit (ADK)

The **Google Agent Development Kit (ADK)** is a powerful framework for building, running, and orchestrating intelligent agents using Google’s Generative AI ecosystem.  
It provides tools and abstractions to make **multi-agent workflows**, **stateful sessions**, and **custom tool integrations** easy to implement — all while keeping the developer experience consistent across different LLM providers.

[Google-ADK-Documentation](https://google.github.io/adk-docs/)

---

##  What is ADK?

At its core, Google ADK simplifies how developers build AI-powered systems that can:
- Use **different models** (Gemini, OpenAI, Ollama, etc.)
- Connect to **custom or built-in tools**
- Run agents **sequentially**, **in parallel**, or in **loops**
- Maintain **state and session management**
- Deploy easily to **web or CLI apps**

---

##  Key Components

| Component | Description |
|------------|--------------|
| **Agent** | The fundamental unit of reasoning — an LLM or workflow that processes input and generates output. |
| **Runner** | Manages execution of agents, including streaming and event handling. |
| **Session** | Stores context and conversation state across multiple agent runs. |
| **Tool** | A callable function or API an agent can use to perform real-world actions. |
| **Workflow** | A structure for chaining multiple agents together (Sequential, Parallel, or Loop). |

---

##  Why Use Google ADK?

-  **Multi-model support** (Gemini, OpenAI, Ollama, etc.)
-  **Tool and plugin integration** (Custom, MCP, CrewAI, etc.)
-  **Reproducible stateful sessions**
-  **Seamless deployment options** via CLI, web, or Streamlit

---



# Basic Agent

### Creating a Basic LLM Agent

This code defines a simple language model–based agent using the `LlmAgent` class from Google ADK. The agent is named **SimpleGeminiAgent** and uses the **gemini-2.5-flash-lite** model.  
It includes a short description to specify its role and an instruction guiding how it should respond.

- `name` provides an identifier for the agent.  
- `model` specifies which LLM backend to use.  
- `description` explains the agent’s purpose.  
- `instruction` defines the tone and clarity of its responses.

The result is a minimal agent that can receive user input and produce concise, model-generated answers.


In [ ]:
from google.adk.agents import LlmAgent

gemini_agent = LlmAgent(
    name="SimpleGeminiAgent",
    model="gemini-2.5-flash-lite",    
    description="A basic AI assistant",
    instruction="Answer clearly and concisely."
)

### Basic Agents using different Models

This example shows how to create an agent that uses OpenAI’s GPT models through the `LiteLlm` wrapper.  
`LiteLlm` is a lightweight adapter that allows ADK to communicate with multiple LLM providers (such as OpenAI or Anthropic) using a consistent interface.

- `LiteLlm(model="gpt-4o")` specifies the OpenAI model to use.  
- Wrapping it inside `LlmAgent` integrates it into the ADK ecosystem, enabling features like session tracking and workflows.  
- The `description` and `instruction` fields define the agent’s purpose and response behavior.

This setup makes it easy to switch between different LLM providers while keeping the rest of the ADK code consistent.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm


openai_agent = LlmAgent(
    name="SimpleGPTAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="A basic AI assistant",
    instruction="Answer clearly and concisely."
)

### Using LiteLLM for Ollama 
Here, it connects to an Ollama model (`ollama_chat/qwen3:8b`) inside an `LlmAgent`.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm


ollama_agent = LlmAgent(
    name="SimpleOllamaAgent",
    model=LiteLlm(model="ollama_chat/qwen3:8b"),
    description="A basic AI assistant",
    instruction="Answer clearly and concisely."
)


This approach lets you swap models easily—by just changing the model name—while keeping the agent logic unchanged.  
Refer to the official documentation for details: [Using Cloud & Proprietary Models via LiteLLM](https://google.github.io/adk-docs/agents/models/#using-cloud-proprietary-models-via-litellm)


# Agents with tools

### Using Inbuilt Tools

Google ADK provides a set of **built-in tools** that let agents perform real-world actions such as searching the web, performing calculations, or retrieving structured data.  
In this example, the `google_search` tool enables an `LlmAgent` to access real-time web data.  

Built-in tools are passed to the agent through the `tools` parameter, and the `instruction` should guide the model on when to use them.

- Tools are added using `tools=[tool_name]` in the agent definition.  

Refer to the official documentation for the full list of available tools:  
[Google ADK Built-in Tools](https://google.github.io/adk-docs/tools/built-in-tools/)


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search

search_agent = LlmAgent(
    name="SearchHelperAgent",
    model="gemini-2.0-flash",  
    description="Agent that uses Google Search for real-time information.",
    instruction="Use the google_search tool when you need current web facts.",
    tools=[google_search],
)


### Creating Custom Tools

Google ADK supports **custom tools**, which are user-defined Python functions that agents can call to perform specific operations or access external logic.  
These tools allow agents to go beyond LLM reasoning and interact with application-specific computations or APIs.

In this example, two simple tools are defined for unit conversions:
- `miles_to_kilometers`: Converts miles to kilometers.  
- `celsius_to_fahrenheit`: Converts Celsius to Fahrenheit.  

They are added to the `tools` parameter of an `LlmAgent`, enabling the model to use them when answering measurement-related queries.

Key points:
- Any Python function with proper type hints and docstrings can serve as a custom tool.  
- Tools are added to an agent with `tools=[...]`.  
- The `instruction` should explicitly mention when the agent should use these tools.  
- **If you want multiple tools to run in parallel**, define them as **`async def`** functions and include guidance in the agent’s `instruction` (e.g., “You may call tools concurrently when applicable”).

Custom tools give you full flexibility to integrate logic, APIs, or computations directly within ADK’s agent framework.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

def miles_to_kilometers(miles: float) -> float:
    """
    Converts miles to kilometers.

    Args:
        miles (float): Distance in miles.

    Returns:
        float: Distance in kilometers.
    """
    return miles * 1.60934


def celsius_to_fahrenheit(celsius: float) -> float:
    """
    Converts Celsius temperature to Fahrenheit.

    Args:
        celsius (float): Temperature in Celsius.

    Returns:
        float: Temperature in Fahrenheit.
    """
    return (celsius * 9 / 5) + 32


unit_converter_agent = LlmAgent(
    name="UnitConverterAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="An agent that performs simple unit conversions.",
    instruction="Use the provided conversion tools to answer measurement-related questions.",
    tools=[miles_to_kilometers, celsius_to_fahrenheit],
)


### Using CrewAI Tools

Google ADK supports integration with **CrewAI tools** through the `CrewaiTool` wrapper.  
This lets agents use existing CrewAI utilities, such as Serper for news or web searches, inside ADK workflows.

Here, a `SerperDevTool` instance is wrapped with `CrewaiTool` and added to an agent via the `tools` parameter.  
The agent’s `instruction` guides when to use it.

CrewAI tools can thus be directly reused within ADK without extra setup, enabling access to powerful third-party capabilities through a unified interface.


In [ ]:

from google.adk import Agent
from google.adk.tools.crewai_tool import CrewaiTool
from crewai_tools import SerperDevTool
from google.adk.models.lite_llm import LiteLlm

# Instantiate the CrewAI tool
serper_tool_instance = SerperDevTool(
    n_results=10,
    save_file=False,
    search_type="news",
)

# Wrap it with CrewaiTool for ADK, providing name and description
adk_serper_tool = CrewaiTool(
    name="InternetNewsSearch",
    description="Searches the internet specifically for recent news articles using Serper.",
    tool=serper_tool_instance
)

# Define the ADK agent
crewai_search_agent = Agent(
    name="crewai_search_agent",
    model=LiteLlm(model="ollama_chat/qwen3:8b"),
    description="Agent to find recent news using the Serper search tool.",
    instruction="I can find the latest news for you. What topic are you interested in?",
    tools=[adk_serper_tool] # Add the wrapped tool here
)

root_agent = crewai_search_agent

### Using MCP Tools

Google ADK supports integration with **Model Context Protocol (MCP)** servers, allowing agents to access external tools or services through a unified interface.  
This enables an agent to communicate with locally or remotely hosted tools such as web search, file operations, or custom APIs.

In this setup:
- `MCPToolset` connects to an MCP server via `StreamableHTTPConnectionParams`.  
- The helper function `get_mcp_tool()` safely initializes this connection.  
- The returned `mcp_tool` is added to the `tools` parameter of an `LlmAgent`.  
- The agent’s `instruction` should indicate when to use the MCP tools.

MCP integration lets ADK agents interact with rich tool ecosystems running outside the current environment, making them capable of more dynamic, data-driven tasks.


In [ ]:
# --- Google ADK Imports ---
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.mcp_tool import MCPToolset, StreamableHTTPConnectionParams


# --- Step 1: Define MCP Tool Getter ---

def get_mcp_tool():
    """
    Safely instantiate MCPToolset to avoid Pydantic schema serialization issues.
    
    This connects the agent to an external MCP server via Streamable HTTP.
    The MCP server can expose tools such as web_search, file_read, or custom logic.
    """
    return MCPToolset(
        connection_params=StreamableHTTPConnectionParams(
            url="http://127.0.0.1:9000/mcp"   # Your FastMCP or MCP server endpoint
        )
    )


# --- Step 2: Create the MCP Tool Instance ---
mcp_tool = get_mcp_tool()


# --- Step 3: Define the Agent using MCP Tool ---
mcp_enabled_agent = LlmAgent(
    name="MCPEnabledAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="An agent connected to an external MCP server to access local or remote tools.",
    instruction=(
        "Use the connected MCP tools whenever external data or file operations are needed. "
        "You can fetch, process, or analyze information through MCP endpoints."
    ),
    tools=[mcp_tool],  # attach the MCP toolset
)


You can refer here for the full MCP-GoogleADK documentation
[Other-MCP-Integration](https://google.github.io/adk-docs/mcp/)

# Graph Workflows

Google ADK 2.0 introduces **graph workflows** as the modern way to model deterministic multi-step agent processes. Instead of relying on the older `SequentialAgent`, `ParallelAgent`, and `LoopAgent` template classes, a graph workflow uses explicit nodes and edges.

A graph node can be a Python function or bound method, an ADK tool, an agent, a human-input request, or a nested workflow. Edges describe how execution moves from one node to the next, including fixed sequences, branches, fan-out/fan-in paths, and approval gates.

> **Requires ADK Python 2.0.0 or newer.** This notebook keeps the deprecated workflow templates at the end for reference, but the examples below are the preferred ADK 2.0 style.

### Route Sequence

A route sequence is the graph equivalent of a sequential workflow. Each node receives data from the previous node and returns an `Event(output=...)` for the next node.

In this example:
- **`normalize_request`** cleans up the raw user request.
- **`extract_topic`** creates a structured dictionary from the normalized text.
- **`planning_agent`** is an agent node that turns the structured request into a final plan.

This pattern is useful when a process has a fixed order and every step should run exactly once.

In [ ]:
from google.adk import Agent, Event, Workflow

GPT_OSS_API_BASE = "http://10.0.10.51:8000/v1"

def content_to_text(node_input):
    """Extract plain text from either a string or a google.genai Content object."""
    if isinstance(node_input, str):
        return node_input
    if hasattr(node_input, "parts"):
        return " ".join(
            part.text for part in node_input.parts
            if getattr(part, "text", None)
        )
    return str(node_input)


def normalize_topic(node_input):
    """Clean the requested topic and store it in session state."""
    raw_text = content_to_text(node_input)
    topic = " ".join(raw_text.strip().split())
    return Event(
        output=topic,
        state={
            "raw_topic": raw_text,
            "topic": topic,
            "workflow_stage": "topic_normalized",
        },
    )


def build_explanation_request(node_input: str):
    """Create a small payload that tells the agent how to explain the topic."""
    explanation_request = {
        "topic": node_input,
        "audience": "beginner",
        "style": "simple and practical",
    }
    return Event(
        output=explanation_request,
        state={
            "audience": explanation_request["audience"],
            "style": explanation_request["style"],
            "workflow_stage": "request_prepared",
        },
    )


explainer_agent = Agent(
    name="explainer_agent",
    model=LiteLlm(model="openai/openai/gpt-oss-20b", api_base=GPT_OSS_API_BASE, api_key="not-needed"),
    mode="single_turn",
    instruction=(
        "You receive a small payload with a topic, audience, and style. "
        "Explain the topic in 3 concise bullet points."
    ),
)

root_agent = Workflow(
    name="simple_topic_explainer_workflow",
    edges=[("START", normalize_topic, build_explanation_request, explainer_agent)],
)


### Conditional Routing

Graph workflows can branch by using a router node that returns an `Event(route=...)`. The workflow maps each route value to the node that should run next.

In this example:
- **`classify_request`** is a method node that chooses a route.
- **`research_agent`**, **`summary_agent`**, and **`support_agent`** are single-turn agent nodes for different request types.

This replaces prompt-only routing with explicit control flow in code.

In [ ]:
from google.adk import Agent, Event, Workflow


class RequestRouter:
    """Class-based router whose method can be used directly as a graph node."""

    def classify_request(self, node_input: str):
        """Return the route that should handle the user's request."""
        text = node_input.lower()
        if "summarize" in text or "summary" in text:
            return Event(route="SUMMARY")
        if "error" in text or "debug" in text or "fix" in text:
            return Event(route="SUPPORT")
        return Event(route="RESEARCH")


research_agent = Agent(
    name="research_agent",
    model="gemini-2.0-flash",
    mode="single_turn",
    instruction="Answer the request as a research task with concise supporting facts.",
)
summary_agent = Agent(
    name="summary_agent",
    model="gemini-2.0-flash",
    mode="single_turn",
    instruction="Summarize the request or supplied content in a short, neutral format.",
)
support_agent = Agent(
    name="support_agent",
    model="gemini-2.0-flash",
    mode="single_turn",
    instruction="Treat the request as a debugging or support task and suggest next steps.",
)

router = RequestRouter()
classify_request = router.classify_request

root_agent = Workflow(
    name="conditional_routing_workflow",
    edges=[
        ("START", classify_request),
        (
            classify_request,
            {
                "RESEARCH": research_agent,
                "SUMMARY": summary_agent,
                "SUPPORT": support_agent,
            },
        ),
    ],
)

### Tool Node in a Graph

A tool node is useful when one step in the graph should call a specific tool directly.

In this simple example:
- **`prepare_numbers`** creates tool arguments: `{"a": 2, "b": 3}`.
- **`add_numbers_tool`** runs as the tool node.
- **`format_result`** turns the tool output into a final response.

The key rule is that the node before a tool must return a dictionary whose keys match the tool function's arguments.


In [ ]:
from google.adk import Event, Workflow
from google.adk.tools import FunctionTool
from google.adk.events import EventActions

def prepare_numbers(node_input):
    """Create the dictionary of arguments expected by the add_numbers tool."""
    return Event(
        output={"a": 2, "b": 3},
        actions=EventActions(
            state_delta={
                "workflow_stage": "numbers_prepared",
                "a": 2,
                "b": 3,
            }
        ),
    )


def add_numbers(a: int, b: int) -> dict:
    """
    Add two numbers.

    Args:
        a: First number.
        b: Second number.

    Returns:
        A dictionary containing the sum.
    """
    return {"a": a, "b": b, "sum": a + b}


def format_result(node_input: dict):
    """Format the tool output as the final graph response and persist the result."""
    result = f"{node_input['a']} + {node_input['b']} = {node_input['sum']}"
    return Event(
        output=result,
        actions=EventActions(
            state_delta={
                "workflow_stage": "result_formatted",
                "sum": node_input["sum"],
                "last_result": result,
            }
        ),
    )


add_numbers_tool = FunctionTool(func=add_numbers)

root_agent = Workflow(
    name="simple_tool_node_workflow",
    edges=[("START", prepare_numbers, add_numbers_tool, format_result)],
)


### Event Data Handling

Graph nodes pass data with `Event` objects. The most important fields are:
- **`output`**: data for the next node.
- **`message`**: user-facing information during workflow execution.
- **`state`**: small values persisted across nodes during the session.

Use `output` for normal node-to-node handoff, `message` when the user should see something, and `state` for compact workflow context such as counters, choices, or flags.

In [1]:
from google.adk import Context, Event, Workflow


async def initialize_attempt_state():
    """Start the workflow with initial state and a user-visible status message."""
    yield Event(
        output="draft itinerary",
        state={"attempts": 0, "approved": False},
        message="Starting itinerary review workflow...",
    )


async def increment_attempt(node_input: str, attempts: int):
    """Increment a persisted attempt counter while passing the draft forward."""
    yield Event(
        output=f"Reviewing: {node_input}",
        state={"attempts": attempts + 1},
    )


async def read_attempt_state(ctx: Context):
    """Read workflow state and emit both machine-readable output and a user message."""
    attempts = ctx.state.get("attempts", 0)
    approved = ctx.state.get("approved", False)
    yield Event(
        output={"attempts": attempts, "approved": approved},
        message=f"Attempt count is now {attempts}.",
    )


root_agent = Workflow(
    name="event_data_handling_workflow",
    edges=[("START", initialize_attempt_state, increment_attempt, read_attempt_state)],
)

### Structured Data with Schemas

For more reliable workflows, graph nodes and agents can use Pydantic schemas to describe the shape of their inputs and outputs.

In this example:
- **`TravelRequest`** defines the expected incoming data.
- **`TravelPlan`** defines the output contract.
- **`itinerary_agent`** is an agent node configured with schema contracts.

Schemas make downstream nodes easier to write because they can rely on named fields instead of parsing free-form strings.

In [ ]:
from datetime import date
from pydantic import BaseModel
from google.adk import Agent, Workflow


class TravelRequest(BaseModel):
    """Structured input expected by the itinerary planning agent node."""

    city: str
    start_date: date
    days: int = 1
    interests: list[str] = []


class TravelPlan(BaseModel):
    """Structured output produced by the itinerary planning agent node."""

    city: str
    title: str
    activities: list[str]


itinerary_agent = Agent(
    name="itinerary_agent",
    model="gemini-2.0-flash",
    mode="single_turn",
    input_schema=TravelRequest,
    output_schema=TravelPlan,
    instruction=(
        "Create a practical itinerary for the requested city. "
        "Return output that conforms to the TravelPlan schema."
    ),
)

root_agent = Workflow(
    name="structured_data_workflow",
    edges=[("START", itinerary_agent)],
)

### Human-in-the-Loop Input

Graph workflows can pause for a human decision with `RequestInput`. This is useful for approval gates, missing information, or asking the user to choose between generated options.

In this example:
- **`draft_agent`** creates a candidate plan.
- **`request_approval`** asks the user whether to approve it.
- **`apply_decision`** continues based on the human response.

The response can be plain text or constrained with a schema when the UI or calling layer can provide structured input.

In [ ]:
from typing import Literal
from pydantic import BaseModel
from google.adk import Agent, Event, Workflow
from google.adk.events import RequestInput


class ApprovalDecision(BaseModel):
    """Expected structure for a human approval response."""

    decision: Literal["approve", "revise", "reject"]
    notes: str = ""


draft_agent = Agent(
    name="draft_agent",
    model="gemini-2.0-flash",
    mode="single_turn",
    instruction="Draft a short implementation plan for the user's requested notebook change.",
)


async def request_approval(node_input: dict):
    """Pause the workflow and ask the human to approve, revise, or reject the draft."""
    message = (
        f"Review this plan: {node_input}\n\n"
        "Respond with a decision: approve, revise, or reject."
    )
    yield RequestInput(
        message=message,
        payload=node_input,
        response_schema=ApprovalDecision,
    )


def apply_decision(node_input: ApprovalDecision):
    """Convert the human decision into the next workflow output."""
    if node_input.decision == "approve":
        return Event(output="Approved. Continue with the workflow.")
    if node_input.decision == "revise":
        return Event(output=f"Revision requested: {node_input.notes}")
    return Event(output=f"Rejected: {node_input.notes}")


root_agent = Workflow(
    name="human_input_workflow",
    edges=[("START", draft_agent, request_approval, apply_decision)],
)

# External Observability using Phoenix-arize

Google ADK integrates with **OpenInference** and **OpenTelemetry** to enable tracing, monitoring, and observability of agent workflows.  
Instrumentation helps track agent calls, tool usage, and performance across distributed systems — useful for debugging, optimization, and analytics.

In this setup:
- **`register()`** initializes a tracer provider under the given project name (`"adk-agent"`).  
- **`GoogleADKInstrumentor`** attaches tracing instrumentation to all ADK agent operations.  
- Once configured, all agent runs, tool invocations, and workflow steps generate trace spans that can be visualized in Phoenix or other OpenTelemetry-compatible dashboards.  

Key points:
- Tracing provides insights into latency, errors, and execution flow across agents and tools.  
- Integrating observability is essential when deploying ADK agents in production or multi-agent systems.  
- The setup works transparently — after instrumentation, no further code changes are needed in your agent logic.

This approach enables comprehensive visibility into the behavior and performance of ADK-based applications.


In [ ]:
from phoenix.otel import register

# Configure the Phoenix tracer
tracer_provider = register(
  project_name="my-llm-app", # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

# Your agent code here...

# Managing Session, and Runner

Google ADK includes built-in components for handling **stateful agent interactions**.  
The combination of `SessionService` and `Runner` enables persistence, context tracking, and state updates across multiple agent runs.

Key components:
- **`SessionService`** manages user sessions and stores conversation state in memory (or persistent storage).  
- **`Session`** represents an active context tied to a specific `app_name`, `user_id`, and `session_id`.  
- **`Runner`** executes agent calls, updates session state automatically, and handles event streaming.  
- **`output_key`** in the agent definition specifies where to store outputs in the session state.

In this example:
- An `LlmAgent` named *Greeter* generates a short greeting and saves it in `state['last_greeting']`.  
- The `Runner` executes the agent while maintaining session continuity.  
- After execution, the updated state reflects the agent’s new output.

This pattern allows ADK agents to maintain memory across turns, enabling multi-step reasoning, context reuse, and workflow continuity.


In [ ]:
import asyncio
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.models.lite_llm import LiteLlm


# --- Define the LLM Agent ---
greeting_agent = LlmAgent(
    name="Greeter",
    model=LiteLlm(model="gpt-4o-mini"),  # or "gpt-4o-mini", etc.
    instruction="Generate a short, friendly greeting.",
    output_key="last_greeting"  # stored in session.state
)


# --- Constants ---
APP_NAME = "state_app"
USER_ID = "user1"
SESSION_ID = "session1"


# --- Main Async Runner Function ---
async def main():
    # Create in-memory session service
    session_service = InMemorySessionService()

    # Create the session
    session = await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )

    print(f"🟢 Initial state: {session.state}")

    # Create runner to execute the agent
    runner = Runner(agent=greeting_agent, app_name=APP_NAME, session_service=session_service)

    # Prepare a user message
    user_message = types.Content(role="user", parts=[types.Part(text="Hello")])

    # Run the agent asynchronously and stream events
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=user_message)

    async for event in events:
        if event.is_final_response():
            response_text = event.content.parts[0].text if event.content and event.content.parts else None
            print(f"🤖 Agent responded: {response_text}")

    # Fetch updated session state
    updated_session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )

    print(f"🟣 Updated state: {updated_session.state}")


# --- Run the Script ---
if __name__ == "__main__":
    asyncio.run(main())




# Running Your ADK Project

Once your agents and workflows are defined, Google ADK provides multiple ways to **run or serve** your application — depending on whether you want an interactive web UI or a command-line workflow.

#### 1. Run via ADK CLI

Use the `adk run` command to launch your agent or workflow directly from the terminal.

```bash
adk run <project_name>
````

This executes the root agent defined in your project’s configuration (e.g., `root_agent` in your main module).

#### 2. Run the Web Interface

To launch the built-in ADK web interface for interactive testing:

```bash
adk web
```

This starts a lightweight web dashboard where you can interact with agents, inspect state, and visualize session activity.

#### 3. Run via Python Script or `uv`

If your project includes a Python entry point (e.g., `app.py`), you can start it using:

```bash
uv run src/<project_name>/app.py
```


#### Notes

* Make sure your environment variables and API keys (for Gemini, OpenAI, etc.) are configured before running.
* When using MCP or CrewAI tools, ensure the corresponding servers (e.g., `fastmcp_server.py`) are running.
* All commands should be executed from the project root unless paths are adjusted.

These commands provide flexible options for testing, debugging, and deploying your ADK-powered applications in different environments.





## Runnable Demo (self-contained)

**Run the cell below on its own** to try Google ADK end-to-end using the local GPT-OSS model configured in this project (`src/playground/agent.py`). No prior cells need to be executed.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Load project .env (this notebook lives in docs/notebooks/)
load_dotenv(Path.cwd().parent.parent / ".env", override=True)

GPT_OSS_API_BASE = os.getenv("GPT_OSS_API_BASE", "http://10.0.10.51:8000/v1")
GPT_OSS_MODEL = os.getenv("GPT_OSS_MODEL", "openai/openai/gpt-oss-20b")

greeting_agent = LlmAgent(
    name="Greeter",
    model=LiteLlm(
        model=GPT_OSS_MODEL,
        api_base=GPT_OSS_API_BASE,
        api_key="not-needed",
    ),
    instruction="Generate a short, friendly greeting.",
    output_key="last_greeting",
)

APP_NAME = "notebook_demo"
USER_ID = "user1"
SESSION_ID = "session1"


async def run_greeting_demo():
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    print(f"Initial state: {session.state}")

    runner = Runner(agent=greeting_agent, app_name=APP_NAME, session_service=session_service)
    user_message = types.Content(role="user", parts=[types.Part(text="Hello!")])

    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=user_message)
    async for event in events:
        if event.is_final_response():
            text = event.content.parts[0].text if event.content and event.content.parts else None
            print(f"Agent responded: {text}")

    updated_session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    print(f"Updated state: {updated_session.state}")


# Jupyter supports top-level await — no prior cells required
await run_greeting_demo()

Initial state: {}
Agent responded: We need to output a short friendly greeting. The persona: agent named "Greeter". So maybe "Hi there! I'm Greeter, happy to help." friendly. Let's keep it concise.
Updated state: {'last_greeting': "Hi! I'm Greeter—nice to meet you! 😊"}


# Deprecated Workflows
### Sequential Workflow

> **Note:** `SequentialAgent`, `ParallelAgent`, and `LoopAgent` examples are kept here for reference, but these workflow patterns have changed in newer Google ADK versions. Treat this section as legacy material.


A **SequentialAgent** in Google ADK allows multiple agents to run one after another in a defined order.  
Each agent’s output is passed automatically as input to the next, enabling multi-step reasoning pipelines.

In this example:
- **`QueryUnderstandingAgent`** rewrites the user’s question into a clear search query.  
- **`WebSearchAgent`** uses the `google_search` tool to retrieve information from the web.  
- **`SummarizerAgent`** compiles the search results into a concise summary.  

These agents are grouped inside a **`SequentialAgent`** named `ResearchWorkflow`, which ensures they execute in sequence.  
This design pattern is ideal for multi-stage processes such as data collection, analysis, and summarization.

Sequential workflows simplify the creation of end-to-end reasoning pipelines where each stage builds upon the previous one.
`

In [ ]:
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import google_search

query_agent = LlmAgent(
    name="QueryUnderstandingAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Rewrites a user question into a clear search query.",
    instruction="Simplify and rewrite the user question to make it ideal for web search."
)

search_agent = LlmAgent(
    name="WebSearchAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Fetches results using Google Search tool.",
    instruction="Use google_search to fetch relevant answers from the web.",
    tools=[google_search],
)

summarizer_agent = LlmAgent(
    name="SummarizerAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Summarizes and compiles the search findings.",
    instruction="Summarize all fetched results into a cohesive short report.",
)

research_workflow = SequentialAgent(
    name="ResearchWorkflow",
    sub_agents=[query_agent, search_agent, summarizer_agent],
    description="A sequential workflow for answering research questions.",
)

### Parallel Workflow

The **ParallelAgent** allows multiple sub-agents to run simultaneously and combine their outputs into a single response.  
This is useful when several analyses or computations can be performed independently on the same input.

In this example:
- **`SentimentAgent`** analyzes the overall sentiment.  
- **`EmotionAgent`** identifies dominant emotions.  
- **`ToneAgent`** determines the tone and reasoning.  
- **`SummaryAgent`** produces a neutral summary.  

All sub-agents execute in parallel within the **`SentimentReviewWorkflow`**, and their results are merged automatically by ADK.

Parallel workflows are efficient for multi-dimensional analysis tasks such as classification, review generation, or summarization where each agent contributes distinct but related insights.


In [ ]:
# --- Imports ---
from google.adk.agents import LlmAgent, ParallelAgent
from google.adk.models.lite_llm import LiteLlm


# --- Step 1: Define Sub-Agents ---

sentiment_agent = LlmAgent(
    name="SentimentAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Analyzes overall sentiment of the text.",
    instruction="Determine whether the text sentiment is positive, negative, or neutral, and explain briefly."
)

emotion_agent = LlmAgent(
    name="EmotionAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Detects dominant emotions expressed in the text.",
    instruction="List any strong emotions present in the text (e.g., joy, anger, sadness, fear)."
)

tone_agent = LlmAgent(
    name="ToneAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Assesses the tone of the text.",
    instruction="Identify the tone (e.g., formal, casual, persuasive, sarcastic) and justify your reasoning."
)

summary_agent = LlmAgent(
    name="SummaryAgent",
    model=LiteLlm(model="gpt-4o-mini"),
    description="Summarizes the content neutrally.",
    instruction="Provide a short, unbiased summary of the text content in one or two sentences."
)


# --- Step 2: Create Parallel Workflow ---

sentiment_review_workflow = ParallelAgent(
    name="SentimentReviewWorkflow",
    sub_agents=[
        sentiment_agent,
        emotion_agent,
        tone_agent,
        summary_agent,
    ],
    description="Runs multiple sentiment analysis agents in parallel and merges their findings.",
)


# --- Step 3: Root Agent (optional) ---
root_agent = sentiment_review_workflow


### Loop Workflow

The **LoopAgent** enables iterative workflows where a group of agents runs repeatedly until a stopping condition is met or a maximum number of iterations is reached.  
This is useful for refinement processes such as code improvement, data cleaning, or document review.

In this example:
- **`CodeRefiner`** updates `state['current_code']` using `state['requirements']`.  
- **`QualityChecker`** evaluates the refined code and sets `state['quality_status']` to `"pass"` or `"fail"`.  
- **`CheckStatusAndEscalate`** (a custom agent) inspects the current state and escalates when quality passes, signaling the loop to stop.  

The **`LoopAgent`** (`CodeRefinementLoop`) runs the three agents in sequence each iteration, up to a maximum of five iterations, or until the escalation condition is triggered.

Loop workflows are ideal for tasks that require repeated refinement, validation, and conditional termination based on evolving agent outputs.


In [ ]:
# Conceptual Code: Iterative Code Refinement
from google.adk.agents import LoopAgent, LlmAgent, BaseAgent
from google.adk.events import Event, EventActions
from google.adk.agents.invocation_context import InvocationContext
from typing import AsyncGenerator

# Agent to generate/refine code based on state['current_code'] and state['requirements']
code_refiner = LlmAgent(
    name="CodeRefiner",
    instruction="Read state['current_code'] (if exists) and state['requirements']. Generate/refine Python code to meet requirements. Save to state['current_code'].",
    output_key="current_code" # Overwrites previous code in state
)

# Agent to check if the code meets quality standards
quality_checker = LlmAgent(
    name="QualityChecker",
    instruction="Evaluate the code in state['current_code'] against state['requirements']. Output 'pass' or 'fail'.",
    output_key="quality_status"
)

# Custom agent to check the status and escalate if 'pass'
class CheckStatusAndEscalate(BaseAgent):
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        status = ctx.session.state.get("quality_status", "fail")
        should_stop = (status == "pass")
        yield Event(author=self.name, actions=EventActions(escalate=should_stop))

refinement_loop = LoopAgent(
    name="CodeRefinementLoop",
    max_iterations=5,
    sub_agents=[code_refiner, quality_checker, CheckStatusAndEscalate(name="StopChecker")]
)
# Loop runs: Refiner -> Checker -> StopChecker
# State['current_code'] is updated each iteration.
# Loop stops if QualityChecker outputs 'pass' (leading to StopChecker escalating) or after 5 iterations.